<a href="https://colab.research.google.com/github/DineshSBhauryal/streamlit-app/blob/main/chat_loa_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install streamlit  # optional (only needed if you plan to use the original version)
!pip install langchain-community
!pip install sentence-transformers
!pip install faiss-cpu
!pip install cohere
!pip install PyPDF2
!pip install numpy
!pip install pandas
!pip install pypdf



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 62.0 MB/s eta 0:00:00


In [ ]:
import os
import getpass

os.environ["COHERE_API_KEY"] = getpass.getpass("Enter Cohere API Key: ")

Enter Cohere API Key: ··········


In [ ]:
import numpy as np
import pandas as pd
import faiss
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from cohere import ClientV2

# Initialize Sentence Transformer model
model_name = "all-MiniLM-L6-v2"  # Adjust model as needed
sentence_transformer_model = SentenceTransformer(model_name)

# Load and process the PDF file
pdf_path = 'Rhea_resume.pdf'
loader = PyPDFLoader(pdf_path)
documents = loader.load()

embeddings = []
documents_text = []
sources = []

# To chunkify the docs
latex_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

for docu in documents:  # one page at a time from the PDF
    docs = latex_splitter.create_documents([docu.page_content])  # splitting into chunks
    for document in docs:
        document_embedding = sentence_transformer_model.encode(document.page_content)
        embeddings.append(document_embedding)
        documents_text.append(document.page_content)
        sources.append("www.rheadata.com")

# Create FAISS index
embedding_dimension = len(embeddings[0])
index = faiss.IndexFlatL2(embedding_dimension)
index.add(np.array(embeddings, dtype='float32'))

# Save index and document details
if not os.path.exists("Vector_Store"):
    os.makedirs("Vector_Store")

df = pd.DataFrame({'documents': documents_text, 'source': sources})
df.to_csv('Vector_Store/docs.csv', index=False)
faiss.write_index(index, 'Vector_Store/vector_db.index')

print("✅ FAISS index and document store created successfully!")


/tmp/ipykernel_2028/1262708446.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ FAISS index and document store created successfully!


In [ ]:
# embeddings[0]

In [ ]:

# --- Q&A Section ---

# Load the FAISS index and CSV (optional reload)
index = faiss.read_index('Vector_Store/vector_db.index')
df = pd.read_csv('Vector_Store/docs.csv')

# Initialize Cohere client
co = ClientV2(api_key=os.environ["COHERE_API_KEY"])

# User query input
query = input("Enter your query: ")

if query:
    query_embedding = sentence_transformer_model.encode(query).reshape(1, -1)
    distances, indices = index.search(query_embedding, k=5)

    threshold = 1.7  # Define threshold for distance match
    print("Distance score:", distances)

    if distances[0][0] > threshold:
        print(" Please ask a relevant question.")
    else:
        combined_similar_documents_content_list = []
        similar_documents_sources = []

        for i in indices[0]:
            similar_document_content = df.loc[i, 'documents']
            combined_similar_documents_content_list.append(similar_document_content)

            similar_document_source = df.loc[i, 'source']
            similar_documents_sources.append(similar_document_source)

        combined_similar_documents_content = ' '.join(combined_similar_documents_content_list)

        cohere_prompt = (
            f"Based on the document content: {combined_similar_documents_content}, "
            f"answer the question: '{query}'"
        )

        # Call Cohere API
        cohere_response = co.chat(
            model="command-a-03-2025",
            messages=[
                {
                "role": "user",
                "content": cohere_prompt
                }
                      ],
            temperature=0.3
        )

        print("\n Bot Response:")
        print(cohere_response.message.content[0].text)

        print("\n Sources:")
        print(list(set(similar_documents_sources)))


Enter your query: education details of Rhea
Distance score: [[1.4257476 1.5949122 1.7933617 1.8389214 1.8626227]]

 Bot Response:
Rhea Rajmanna is currently pursuing a degree in **General Engineering with an interest in Computer Science** at **Virginia Polytechnic Institute and State University (Virginia Tech)** in Blacksburg, VA. She is expected to graduate in **May 2027**. Prior to this, she attended **Chantilly High School** in Chantilly, VA, where she earned an **Advanced Diploma** with a **weighted GPA of 4.46/4.00** from **August 2020 to June 2024**. 

**Relevant Coursework** at Virginia Tech includes:  
- Software Design and Data Structures  
- Discrete Mathematics  
- Multivariable Calculus  
- Introduction to Linear Algebra  
- Foundations of Engineering  
- Foundations of Physics  
- Technical Writing  

In high school, she completed advanced courses such as:  
- AP BC Calculus (Score: 5)  
- AP AB Calculus (Score: 5)  
- AP Research (Score: 5)  
- AP Computer Science A (Scor